# How much variance does one component carry per family?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [16]:
import json
import time

import numpy as np
import plotly.express as px
import polars as pl
from IPython.display import Markdown, display
from sklearn.decomposition import PCA

In [17]:
def get_raw_dir():
    import os
    from pathlib import Path
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "kaggle" / "raw").exists():
            return current / "kaggle" / "raw"
        current = current.parent
    return Path("../../kaggle/raw")

raw_dir = get_raw_dir()
df = pl.scan_csv(raw_dir / "train_transaction.csv", infer_schema_length=10000, null_values=[""]).collect()
v_cols = [c for c in df.columns if c.startswith("V")]

# Rebuild the subgroups and representatives
def get_references_dir():
    import os
    from pathlib import Path
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "references").exists():
            return current / "references"
        current = current.parent
    return Path("../../references")

references_dir = get_references_dir()
with open(references_dir / "column-groups-v.json", "r") as f:
    col_groups_json = json.load(f)

subgroups = []
assigned_cols = set()
for block in col_groups_json['blocks']:
    for group in block['groups']:
        subgroups.append(group)
        assigned_cols.update(group)

unassigned = set(v_cols) - assigned_cols
for u in unassigned:
    subgroups.append([u])

representatives = []
for group in subgroups:
    if len(group) == 1:
        representatives.append(group[0])
    else:
        uniques = [(c, df[c].n_unique()) for c in group]
        best_col = max(uniques, key=lambda x: x[1])[0]
        representatives.append(best_col)


In [18]:
df_pd = df[v_cols].to_pandas()
df_imputed = df_pd.fillna(df_pd.median())

start_time = time.time()
explained_variances = []
pca_exceptions = []
multi_groups = 0
holds_80 = 0

for group in subgroups:
    if len(group) == 1:
        continue
    
    multi_groups += 1
    pca = PCA(n_components=1)
    pca.fit(df_imputed[group])
    var = pca.explained_variance_ratio_[0]
    explained_variances.append(var)
    
    if var > 0.8:
        holds_80 += 1
    else:
        pca_exceptions.append({
            "Group": "+".join(group) if len(group) <= 4 else f"{group[0]}+{group[1]}+{group[2]}…+{group[-1]}",
            "Columns": len(group),
            "Explained variance": var
        })

runtime = time.time() - start_time


### PCA Variance Distribution

In [19]:
fig = px.histogram(x=explained_variances, nbins=20, 
                   title='PCA Explained Variance Ratio across V-Column Groups',
                   labels={'x': 'Explained Variance by First Component', 'y': 'Number of Groups'})
fig.update_yaxes(title_text='Number of Groups')
fig.add_vline(x=0.8, line_dash='dash', line_color='red', annotation_text='80% variance')
fig.update_layout(showlegend=False, template='plotly_white')
fig.show()


#### PCA Results Summary

In [20]:
summary = (
    f"| Metric | Value |\n"
    f"| --- | ---: |\n"
    f"| Columns in | {len(v_cols)} |\n"
    f"| Components out | **{len(subgroups)}** |\n"
    f"| Runtime | {runtime:.1f} s |\n"
    f"| Explained variance, median | **{np.median(explained_variances):.3f}** |\n"
    f"| … minimum | {min(explained_variances):.3f} |\n"
    f"| Multi-column groups whose first component holds > 80% | **{holds_80} of {multi_groups} ({holds_80/multi_groups*100:.0f}%)** |\n\n"
    f"### Exceptions\n\n"
)

df_ex = pl.DataFrame(pca_exceptions).sort("Explained variance")
summary += df_ex.to_pandas().to_markdown(index=False)
display(Markdown(summary))

| Metric | Value |
| --- | ---: |
| Columns in | 339 |
| Components out | **129** |
| Runtime | 0.6 s |
| Explained variance, median | **0.946** |
| … minimum | 0.734 |
| Multi-column groups whose first component holds > 80% | **92 of 93 (99%)** |

### Exceptions

| Group               |   Columns |   Explained variance |
|:--------------------|----------:|---------------------:|
| V108+V109+V110+V114 |         4 |             0.733727 |

#### Cost of PCA

PCA requires numerical inputs without missing values. Because missingness serves as a structural signal in this dataset, imputing the values removes this dimension. The resulting components cannot express missingness natively.